In [ ]:
#lOAD PACKAGES
import sys
import os
sys.path.append(os.path.join("..", "scripts", "analysis"))
import pandas as pd
import plotly.express as px
import plotly.subplots as sp
# Import the processing module from the same folder

from processing import load_solutions, apply_conservative_classification, combine_solutions, calculate_supply_demand, calculate_reserve
from plotting import plot_supply_demand, plot_reserve, plot_reserve_by_fieldy
# import processing 
# from pivottablejs import pivot_ui
G_save = False

dim = (1000,500)
g_BLUE = "1616A7"
g_GREY = "#7F7F7F"

legend_attr = dict(
    x=0.5,
    y=-0.25,
    yanchor="bottom",
    xanchor="center",
    orientation="h"
)

In [ ]:


ss = [
    # {'solution_folder': f"RTS-GMLC_v3.1.1s", 'VLGEN': 30, 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v4.1.1s", 'VLGEN': 30, 'model_type' : 'e-reserve'},
    {'solution_folder': f"RTS-GMLC_v32.3s", 'model_type' : 'envelope'},
    {'solution_folder': f"RTS-GMLC_v32.1s", 'model_type' : 'e-reserve'},
    # {'solution_folder': f"RTS-GMLC_v_s1.2s", 'VLGEN': 1e3, 'model_type' : 'stochastic'}
]
days = [131] 
#days = [49, 41, 131, 144]
s_uc = []
s_ed = []

solution_keys = ['demand','generation','storage','reserve','energy_reserve', 'storage_parameters']
for sol in ss:
    s = sol['solution_folder']
    s_uc_ = load_solutions("s_uc", os.path.join("..", "output", s), days, solution_keys = solution_keys,  model_type = sol['model_type'], solution_id = s)
    if sol['model_type'] != 'stochastic':
        s_ed_ = load_solutions("s_ed", os.path.join("..", "output", s), days, solution_keys = solution_keys, model_type = sol['model_type'], solution_id = s)
    else:
        s_ed_ = load_solutions("s_suc", os.path.join("..", "output", s), days, solution_keys = solution_keys, model_type = sol['model_type'], solution_id = s)
    s_uc.append(s_uc_)
    s_ed.append(s_ed_)

s_uc = combine_solutions(s_uc)
s_ed = combine_solutions(s_ed)


for k,v in s_uc.items():
    if 'µ' in v.columns:
        s_uc[k] = apply_conservative_classification(s_uc[k])
for k,v in s_ed.items():
    if 'µ' in v.columns:
        s_ed[k]= apply_conservative_classification(s_ed[k])



In [ ]:
# Define group_by and calculate supply, demand, and reserves
group_by = ['configuration', 'day', 'model_type']
supply_uc = {}
demand_uc = {}
supply_ed = {}
demand_ed = {}

# Calculate supply and demand for UC and ED
fields = ['hour', 'resource'] + group_by
supply_uc, demand_uc = calculate_supply_demand(s_uc, fields)
supply_ed, demand_ed = calculate_supply_demand(s_ed, fields + ['iteration'])

# Handle reserves calculation
reserve_uc = calculate_reserve(s_uc['reserve'], None, ['hour', 'resource'] + group_by)

if len(s_uc.get('energy_reserve', [])) > 0:
    s_uc_reserve = s_uc['energy_reserve'][s_uc['energy_reserve']['hour'] == s_uc['energy_reserve']['hour_i']]
    s_uc_reserve = s_uc_reserve.drop('hour_i', axis=1)
    s_uc_reserve = s_uc_reserve.rename(columns={
        'energy_reserve_up_MW': 'reserve_up_MW',
        'energy_reserve_down_MW': 'reserve_down_MW'
    })
    reserve_uc = pd.concat([calculate_reserve(s_uc_reserve, None, ['hour', 'resource'] + group_by),reserve_uc], axis=0) 

# Calculate commitments
commit_uc = s_uc['generation'].groupby(['resource', 'configuration', 'day', 'hour'])['commit'].sum().reset_index()
commit_ed = s_ed['generation'].groupby(['resource', 'configuration', 'day', 'iteration', 'hour'])['commit'].sum().reset_index()


In [ ]:
# group_by_big= [:configuration, :day]
# gcdr_reserve_uc = combine(groupby(reserve_uc, [:configuration, :day, :resource]), [:reserve_up_MW,:reserve_down_MW]  .=> sum .=> [:reserve_up_MW,:reserve_down_MW])
# # combine(groupby(reserve_uc, union(group_by_big, [])), [:reserve_up_MW,:reserve_down_MW]  .=> sum .=> [:reserve_up_total_MW,:reserve_down_total_MW])
# gcdr_reserve_uc = leftjoin!(gcdr_reserve_uc,
# combine(groupby(reserve_uc, union(group_by_big, [:configuration, :day])), [:reserve_up_MW,:reserve_down_MW]  .=> sum .=> [:reserve_up_total_MW,:reserve_down_total_MW]),
# on = group_by_big
# )

# gcdr_reserve_uc.reserve_down_relative = gcdr_reserve_uc.reserve_down_MW ./ gcdr_reserve_uc.reserve_down_total_MW
# gcdr_reserve_uc.reserve_up_relative = gcdr_reserve_uc.reserve_up_MW ./ gcdr_reserve_uc.reserve_up_total_MW
# transform!(gcdr_reserve_uc, :configuration .=> ByRow(x -> parse_configuration_to_mu(x)) .=> :mu)
# sort!(gcdr_reserve_uc, :mu)
# ;
# gcdi_KPI_adequacy = calculate_adecuacy_gcdi_KPI(s_ed, s_uc)
# gcd_KPI_adequacy = calculate_adecuacy_gcd_KPI(gcdi_KPI_adequacy)
# ;

In [ ]:
# day_ = 0
iteration_ = 'demand_1'
config_ = supply_uc.configuration.unique()[1]
# config_2 = supply_uc.configuration.unique()[-1]
model_type_ = 'conservative'
day_ = days[0]

### UC

In [ ]:


# supply_uc_ = supply_uc[(supply_uc.day == day_) & (supply_uc.model_type == model_type_)]
# demand_uc_ = demand_uc[(demand_uc.day == day_) & (demand_uc.model_type == model_type_)]
# reserve_uc_ = reserve_uc[(reserve_uc.day == day_) & (reserve_uc.model_type == model_type_)]
# plot_supply_demand(supply_uc_, demand_uc_, str(config_))

In [ ]:
# plot_reserve(reserve_uc_)


In [ ]:
maps = {
    'model_type': {'envelope': 'dynamic'},
    'resource': {'system_loss_of_generation_ED': 'Curtailment', 'hydro_reservoir': 'Hydro dam', 'battery': 'Battery', 'NUCLEAR':'Nuclear'}
}

supply_uc_ = supply_uc[(supply_uc.day == day_)].replace(maps)
demand_uc_ = demand_uc[(demand_uc.day == day_)].replace(maps)
reserve_uc_ = reserve_uc[(reserve_uc.day == day_)].replace(maps)

In [ ]:
supply_uc_ = supply_uc_[supply_uc_.resource != 'net_generation']
reserve_uc_ = reserve_uc_[reserve_uc_.resource != 'Nuclear']
reserve_uc_ = reserve_uc_[reserve_uc_.resource != 'system']

In [ ]:
supply_uc_.resource.unique()
# demand_uc_.resource.unique()
# reserve_uc_.resource.unique()

In [ ]:

def update_background(fig, legend_attr=legend_attr, dim=dim, showlegend=True):
    fig.update_layout(
        plot_bgcolor="rgba(0,0,0,0)",
        # yaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True),
        # xaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True),
        width=dim[0],
        height=dim[1],
        showlegend=showlegend,
        autosize=False,
        legend=legend_attr  # Include legend attributes
    )
    config = dict(showgrid=False, showticklabels=True, showline=True, linecolor="grey", mirror=True, gridwidth=0.11, gridcolor="grey")
    # fig.for_each_yaxis(lambda yaxis: yaxis.update(**config))
    # fig.for_each_xaxis(lambda xaxis: xaxis.update(**config))
    fig.update_yaxes(**config)
    
    fig.update_xaxes(**config)
    

In [ ]:
latex_textwidth_pt = 516.0
scale = 1
dim = (latex_textwidth_pt * scale,latex_textwidth_pt * scale*0.73)
# Set page width large enough to avoid legend wrapping
# plt.rcParams['figure.figsize'] = [20, 10]
legend_attr = dict(
    x=1,
    y=1,
    yanchor="top",
    xanchor="left",
    orientation="v",
    tracegroupgap=0,
    itemwidth=30,  
    # borderwidth=1,
    # itemsizing='constant'
    # valign='top',
    # tracegroupgap=2,
        # itemsizing='constant',
            # itemsizing='trace',
        # traceorder='normal',
    # font=dict(size=5),
)
fig = plot_supply_demand(supply_uc_, demand_uc_, title=None, column_key='model_type')
# update_background(fig, legend_attr=legend_attr, dim=dim, showlegend=True)
fig.update_layout(
    legend=legend_attr,
    showlegend=True,
    # autosize=False, 
    # grid = {'rows': 3, 'columns': 2, 'pattern': 'independent'},
    width=dim[0],
    height=dim[1],
    yaxis_domain=[0.7, 1.0],
    yaxis3_domain=[0.35, 0.65],
    yaxis5_domain=[0.0, 0.3],
    yaxis2_domain=[0.7, 1.0],
    yaxis4_domain=[0.35, 0.65],
    yaxis6_domain=[0.0, 0.3],
)
n = 4
fig.update_layout(margin=dict(l=n, r=n+30, t=n+30, b=n))
fig.show()

In [ ]:
if G_save:
    fig.write_image("dispatch_schedule.pdf", width=dim[0], height=dim[1])
    # fig.write_image("dispatch_schedule.png", width=dim[0], height=dim[1])

In [ ]:


fig = plot_reserve(reserve_uc_, title = None, column_key='model_type')

fig.update_layout(
    legend=legend_attr,
    showlegend=True,
    # autosize=False, 
    # grid = {'rows': 3, 'columns': 2, 'pattern': 'independent'},
    width=dim[0],
    height=dim[1],
    yaxis_domain=[0.7, 1.0],
    yaxis3_domain=[0.35, 0.65],
    yaxis5_domain=[0.0, 0.3],
    yaxis2_domain=[0.7, 1.0],
    yaxis4_domain=[0.35, 0.65],
    yaxis6_domain=[0.0, 0.3],
)
n = 4
fig.update_layout(margin=dict(l=n, r=n+30, t=n+30, b=n))
fig.show()

In [ ]:
if G_save:
    fig.write_image("reserve_schedule.pdf", width=dim[0], height=dim[1])
    # fig.write_image("reserve_schedule.png", width=dim[0], height=dim[1])